In [ ]:


from glob import glob
import numpy as np    
import pandas as pd
import matplotlib.pyplot as plt
import scipy.optimize as op
import os

def Collatz(Tf, Ci, APAR, Vcmax25, Ps, alf=0.085):
    '''Photosynthesis for C3 plant.
    Reference: (Collatz et al., 1991)
    '''
    
    R = 8.314e-3    # [kJ K-1 mol-1] 
    O = Ps * 0.21    # [Pa]
    Pi = Ci*1e-6 * Ps
    
    # Temperature correction
    item = (Tf-298.15)/10
    KC25 = 30    # [Pa]
    KCQ10 = 2.1    # [-]
    KO25 = 30000    # [Pa]
    KOQ10 = 1.2    # [-]
    tao25 = 2600    # [Pa]
    taoQ10 = 0.57    # [-]
    KC = KC25 * KCQ10**item    # [Pa]
    KO = KO25 * KOQ10**item    # [Pa]
    K = KC*(1+O/KO)    # [Pa] 
    tao = tao25 * taoQ10**item    # [Pa]
    GammaS = O / (2*tao)    # [Pa]
    VcmaxQ10 = 2.4    # [-]
    Vcmax_o = Vcmax25 * VcmaxQ10**item    # [umol m-2 s-1]
    Vcmax = Vcmax_o / (1+np.exp((-220+0.703*Tf)/(R*Tf)))    # [umol m-2 s-1] 
    Tleaf_C = Tf - 273.15
    Rd_o = 0.015 * Vcmax
    Rd = Rd_o / (1 + np.exp(1.3 * (Tleaf_C - 55)))
    
        
    # Three limiting states
    JC = Vcmax * (Pi-GammaS)/(Pi+K)
    JE = alf * APAR * (Pi-GammaS)/(Pi+2*GammaS) 
    JS = Vcmax / 2
   
    
    # Colimitation (not the case at canopy level according to DePury and Farquhar)
    a = 0.98
    b = -(JC+JE)
    c = JC * JE
    JCE = (-b + np.sign(b) * np.sqrt(b**2 - 4*a*c))/(2*a)
    JCE = np.real(JCE)
    a = 0.95
    b = -(JCE+JS)
    c = JCE * JS
    JCES = (-b + np.sign(b) * np.sqrt(b**2 - 4*a*c))/(2*a)
    JCES = np.real(JCES)
    
    # Net assimilation
    
    An = JCES - Rd
    
    
    return An,JC,JE,Vcmax,GammaS, Rd
    
def resi(p, x, y):
    tleaf = x[0]
    ci = x[1]
    apar = x[2]
    press = x[3]
    Vcmax = p[0]  
    An = Collatz(tleaf, ci, apar, Vcmax, press)[0]
    return y - An  # 残差


## Set your folders
names = ['20190627SY','20190709SY','20190728SY','20190803SY','20190810SY','20190818SY','20190823SY','20190907SY','20200615SY']

for name in names:
    input = '../data/soybean2019/raw%s' % name
    output = '../data/soybean2019_result/Collatz_raw%s' % name
    
    os.makedirs(output,exist_ok=True)
    files = glob('%s/*.xlsx' % input)
    files.sort()
    # result = open('%s/result.csv' % output,'w')
    # result.write('file,Vcmax25,J25,Rd25,J25/Vcmax25,R2,RMSE\n')
    file_names = []
    Vcmax25s_list = []
    Vcmaxs_list = []
    Rds_list = []
    GammaSs_list = []
    R2_list =[]
    RMSE_list = [] 
    Ao_list = [] 
    Ann_list = [] 

    for file in files:
        print(file)
        xlsx = pd.read_excel(file,skiprows=14)
        A = xlsx['A'][1:].astype(np.float32)
        Ci = xlsx['Ci'][1:].astype(np.float32)
        Tf = xlsx['Tleaf'][1:].astype(np.float32) + 273.15    # [K]
        APAR = xlsx['Qin'][1:].astype(np.float32) * 0.86
        Ps = xlsx['Pa'][1:].astype(np.float32) * 1000
        msk = Ci>0
        A = A[msk]
        Ci = Ci[msk]
        Tf = Tf[msk]
        APAR = APAR[msk]
        Ps = Ps[msk]

        # Fit Vcmax25 
        p0 = [100, 150, 1]  # Initial guesses for Vcmax25 
        p_opt = op.leastsq(resi, [100], args=([Tf, Ci, APAR, Ps], A))[0]
        Vcmax25 = p_opt[0]
        An, JC, JE, Vcmax, GammaS, Rd = Collatz(Tf, Ci, APAR, Vcmax25, Ps)
        Rd = Rd.mean()
        R2 = 1 - np.sum((A - An)**2) / np.sum((A - np.mean(A))**2)
        RMSE = np.sqrt(np.mean((A - An) ** 2))
        Ao = A.mean()
        Ann = An.mean()
        Vcmax = Vcmax.mean()
        GammaS = GammaS.mean()
        file_names.append(file.split('\\')[-1]+',')
        Vcmax25s_list.append(Vcmax25) 
        Rds_list.append(Rd)
        GammaSs_list.append(GammaS)
        Vcmaxs_list.append(Vcmax) 
        R2_list.append(R2)
        RMSE_list.append(RMSE)
        Ao_list.append(Ao)
        Ann_list.append(Ann)

    df0 = pd.DataFrame(file_names, columns=['filenames'])
    df1 = pd.DataFrame(Vcmax25s_list, columns=['Vcmax25'])
    
   
    df4 = pd.DataFrame(Rds_list, columns=['Rd'])
    df6 = pd.DataFrame(Vcmaxs_list, columns=['Vcmax'])
    

    df8 = pd.DataFrame(GammaSs_list, columns=['GammaS'])
    

    df11 = pd.DataFrame(R2_list, columns=['R2'])
    df12 = pd.DataFrame(RMSE_list, columns=['RMSE'])
    df13 = pd.DataFrame(Ao_list, columns=['A'])
    df14 = pd.DataFrame(Ann_list, columns=['An'])
  
    #df_combined = pd.concat([df0, df1, df2, df3, df6, df7,df11,df12,df13,df14], axis=1)
    dfs = [df0, df1, df6, df4,df8,df11, df12, df13, df14]
    df_combined = pd.concat([df.reset_index(drop=True) for df in dfs], axis=1)

    filename = output + '/result.csv'
    # print(filename)
    df_combined.to_csv(filename,index=False)  
        